# 00 · Setup & SQL Foundations

Welcome to the **SQL Zero-to-Hero Bootcamp**! 🎓

You will learn SQL by *doing*. Every notebook mixes short explanations with
runnable examples and hands-on exercises. You write SQL directly in cells using
the `%%sql` magic and see results instantly as tables.

## How this works
- The whole course runs on a small SQLite database — no server to install.
- We use [JupySQL](https://jupysql.ploomber.io/) so cells that start with
  `%%sql` are pure SQL. A single-line query can use inline `%sql SELECT ...`.
- Results are returned as pandas DataFrames, so they render as nice tables.

## Before you start
1. Make sure you built the database (from a terminal at the project root):
   ```bash
   uv run python data/build_database.py
   ```
2. Then run the setup cell below in every notebook (it's always the first cell).

In [ ]:
# ▶ Run this cell first. It loads JupySQL and connects to the SQLite database.
%load_ext sql
from sqlalchemy import create_engine
import os

# Works whether the notebook's working dir is the repo root or notebooks/
db_path = 'data/retail.db' if os.path.exists('data/retail.db') else '../data/retail.db'
engine = create_engine(f'sqlite:///{db_path}')

%config SqlMagic.autopandas = True      # results come back as pandas DataFrames
%config SqlMagic.displaycon = False
%config SqlMagic.feedback = 0
%config SqlMagic.displaylimit = 100

%sql engine
print('Connected to', db_path)

## The database: a small retail company

Everything you query is this fictional shop. Understanding the tables now will
make every later module easier.

| Table | What it holds | Key columns |
|-------|---------------|-------------|
| `categories`  | product categories | `category_id` |
| `suppliers`   | who supplies products | `supplier_id` |
| `products`    | the catalog | `product_id`, `category_id`, `supplier_id`, `unit_price` |
| `customers`   | people who buy | `customer_id`, `country`, `signup_date` |
| `employees`   | staff; `manager_id` points at another employee | `employee_id`, `manager_id` |
| `orders`      | one row per order | `order_id`, `customer_id`, `employee_id`, `order_date`, `status` |
| `order_items` | products within an order | (`order_id`, `product_id`), `quantity`, `unit_price` |

**How they relate**

```
categories 1───∞ products ∞───1 suppliers
customers  1───∞ orders   ∞───1 employees (employee who took the order)
orders     1───∞ order_items ∞───1 products
employees  1───∞ employees  (manager_id is a self reference)
```

Let's look at the tables SQLite knows about:

In [ ]:
%%sql
SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name;

Peek at a few rows of `products`:

In [ ]:
%%sql
SELECT * FROM products LIMIT 5;

And a few customers:

In [ ]:
%%sql
SELECT customer_id, first_name, last_name, country FROM customers LIMIT 5;

## Your very first query

`SELECT` retrieves data. `*` means "all columns". `LIMIT` caps the row count.

In [ ]:
%%sql
SELECT * FROM categories;

## Think in *sets*, not loops

The single biggest mindset shift for engineers coming from imperative code: SQL
is **declarative and set-based**. You don't tell the database *how* to loop over
rows — you *describe the result you want* and the query planner figures out how
to produce it. A `WHERE` clause isn't an `if` inside a `for`; it's a filter
applied to the whole set at once. Writing row-by-row logic (cursors, procedural
loops) is almost always slower and harder to read than one set-based statement.

## How SQL is actually processed (logical order)

You *write* a query starting with `SELECT`, but the database *evaluates* the
clauses in a different **logical order**. Knowing this explains many "why doesn't
this work?" moments (e.g. why you can't use a `SELECT` alias in `WHERE`, but can
in `ORDER BY`):

```
1. FROM / JOIN      pick the tables and combine them
2. WHERE            filter individual rows
3. GROUP BY         collapse rows into groups
4. HAVING           filter the groups
5. SELECT           choose/compute the output columns (aliases created here)
6. DISTINCT         remove duplicate output rows
7. ORDER BY         sort the final result
8. LIMIT / OFFSET   keep a slice
```

Because `WHERE` (step 2) runs *before* `SELECT` (step 5), a column alias defined
in `SELECT` doesn't exist yet in `WHERE`. But `ORDER BY` (step 7) runs *after*
`SELECT`, so ordering by an alias works. Keep this list handy — it's the mental
model behind everything that follows.

### ✅ You're set up!
You connected to the database, listed its tables, ran your first `SELECT`, and
now understand SQL's set-based, declarative model and its logical evaluation
order.

**Next:** `01_select_basics.ipynb` — choosing columns, aliases, `DISTINCT`, and
calculated columns.